# LF hypertrophy pilot — GSE294458 (within-fibroblast STATE signature)

**Arm A / Patent-51 analog.** Pilot single-cell analysis of human ligamentum flavum (LF) hypertrophy to build a **within-fibroblast STATE contrast** (activated-fibroblast / myofibroblast vs resting fibroblast) as the primary disease signature for downstream LINCS L1000 signature-reversal repurposing.

**Why STATE, not disease-STAGE.** The companion disc arm showed that within-cell *state* contrasts replicate across cohorts and platforms while disease-*stage* (case vs control) contrasts do not. GSE294458 is **1 hypertrophic (HLF) vs 1 control (NLF) donor**, so a case-vs-control contrast here is badly confounded by donor. We therefore build the signature from the within-tissue myofibroblast-vs-resting fibroblast contrast, pooling cells across both donors, and treat the HLF-vs-NLF contrast as a fragile secondary check only.

> **PILOT — no replication claim.** One cohort, n=1 vs 1. Nothing here advances to a filing until the signature is reproduced in a second, independent LF cohort (Ham/Korea U 3-sample; Zhang/SMU 5v5 — both pending author data requests). Illustrative-weight modeling, not measured pharmacology. Not medical or legal advice.

Runtime: Colab (CPU is fine for ~11k cells; GPU not required).

In [ ]:
# 1. Install dependencies (Colab)
!pip -q install "scanpy>=1.10" leidenalg python-igraph 2>/dev/null
# harmonypy is optional (batch/donor integration); safe if it fails
!pip -q install harmonypy 2>/dev/null
print("done")

In [ ]:
# 2. Imports and settings
import os, glob, tarfile, urllib.request, shutil
import numpy as np
import pandas as pd
import scanpy as sc

sc.settings.verbosity = 1
sc.settings.figdir = "figs"
os.makedirs("figs", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("out", exist_ok=True)
print("scanpy", sc.__version__)

## Step 1 — Download GSE294458 from the GEO FTP

The GEO series supplementary directory holds the processed matrices. The exact filenames are not known ahead of time, so the next cell lists the directory, downloads everything, and unpacks any TAR archives. **Inspect the printed file list**, then adjust the loader in Step 2 if the filename patterns differ from what is assumed.

In [ ]:
# 3. Fetch supplementary files from the GEO FTP
BASE = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE294nnn/GSE294458/suppl/"

def list_ftp_dir(url):
    html = urllib.request.urlopen(url, timeout=60).read().decode("utf-8", "ignore")
    import re
    names = re.findall(r'href="([^"?/][^"]*)"', html)
    return [n for n in names if not n.startswith(("http", "/"))]

files = list_ftp_dir(BASE)
print("Supplementary files listed:")
for f in files:
    print("  ", f)

for f in files:
    dst = os.path.join("data", f)
    if not os.path.exists(dst):
        print("downloading", f)
        urllib.request.urlretrieve(BASE + f, dst)

# unpack any tar archives
for tarpath in glob.glob("data/*.tar"):
    print("extracting", tarpath)
    with tarfile.open(tarpath) as tf:
        tf.extractall("data")

print("\nAll files under data/ after unpack:")
for p in sorted(glob.glob("data/**/*", recursive=True)):
    print("  ", p)

## Step 2 — Load into a single AnnData and label donors

`GSE294458_RAW.tar` unpacks to **GEO-style prefixed triplets** (e.g. `GSM8906515_LSS_matrix.mtx.gz` + `_barcodes.tsv.gz` + `_features.tsv.gz`), not bare `matrix.mtx` files, so `sc.read_10x_mtx` cannot read them directly. The loader below groups each triplet by its filename prefix, reads the Matrix Market file, transposes to cells x genes, and pulls gene symbols from column 2 of the features file. It labels each cell's sample as **HLF** (hypertrophic; GSM8906515 / "LSS") or **NLF** (control; GSM8906516 / "LDH") from the prefix. It also still handles a combined `.h5ad` or 10x `.h5` if present.

If the printed filenames from Step 1 use a different token for case/control, edit `infer_group()` accordingly.

In [ ]:
# 4. Load matrices -> AnnData  (handles GEO-style PREFIXED mtx/tsv triplets)
# GSE294458_RAW.tar unpacks to files like GSM8906515_LSS_matrix.mtx.gz +
# _barcodes.tsv.gz + _features.tsv.gz (NOT bare matrix.mtx), so sc.read_10x_mtx
# will not find them. We group each triplet by its filename prefix and load it.
import re
import scipy.io, scipy.sparse as sp
import anndata as ad

def infer_group(name):
    u = name.upper()
    if "HLF" in u or "LSS" in u or "GSM8906515" in u:
        return "HLF"
    if "NLF" in u or "LDH" in u or "GSM8906516" in u:
        return "NLF"
    return "UNKNOWN"

def _read_tsv(path):
    return pd.read_csv(path, header=None, sep="\t", compression="infer")

def load_prefixed_triplet(mtx_path):
    d = os.path.dirname(mtx_path) or "."
    base = os.path.basename(mtx_path)
    prefix = base[:re.search(r"matrix\.mtx", base, re.I).start()]
    sibs = [f for f in os.listdir(d) if f.startswith(prefix)]
    bfile = next(f for f in sibs if "barcode" in f.lower())
    ffile = next(f for f in sibs if ("feature" in f.lower() or "gene" in f.lower())
                 and "mtx" not in f.lower())
    M = scipy.io.mmread(mtx_path).tocsr()              # features x cells
    barcodes = _read_tsv(os.path.join(d, bfile)).iloc[:, 0].astype(str).values
    feats = _read_tsv(os.path.join(d, ffile))
    genes = (feats.iloc[:, 1] if feats.shape[1] > 1 else feats.iloc[:, 0]).astype(str).values
    A = ad.AnnData(X=sp.csr_matrix(M.T))               # -> cells x genes
    A.obs_names = barcodes
    A.var_names = genes
    A.var_names_make_unique()
    A.obs["sample"] = infer_group(prefix)
    return A

h5ads = glob.glob("data/**/*.h5ad", recursive=True)
h5s   = glob.glob("data/**/*.h5",   recursive=True)
mtxs  = sorted(glob.glob("data/**/*matrix.mtx*", recursive=True))

adatas = []
if h5ads:
    for p in h5ads:
        a = sc.read_h5ad(p); a.obs["sample"] = infer_group(p); adatas.append(a)
elif h5s:
    for p in h5s:
        a = sc.read_10x_h5(p); a.var_names_make_unique()
        a.obs["sample"] = infer_group(p); adatas.append(a)
elif mtxs:
    print("MTX triplets found:")
    for m in mtxs: print("  ", m)
    adatas = [load_prefixed_triplet(m) for m in mtxs]
else:
    raise FileNotFoundError(
        "No .h5ad / .h5 / *matrix.mtx* found under data/. "
        "Inspect the Step 1 file list and adjust the globs.")

if len(adatas) == 1 and adatas[0].obs["sample"].nunique() > 1:
    adata = adatas[0]
else:
    adata = ad.concat(adatas, join="outer", label="batch", index_unique="-")

adata.obs_names_make_unique()
print(adata)
print(adata.obs["sample"].value_counts())
assert set(adata.obs["sample"].unique()) & {"HLF", "NLF"}, "Sample labels not assigned - edit infer_group()."

## Step 3 — QC and filtering (incl. erythrocyte removal)

Standard 10x QC (min genes, mito fraction) plus an explicit **hemoglobin-fraction filter to remove red blood cells**. LF surgical tissue carries substantial blood; erythrocytes are gene-poor and near-pure hemoglobin, and if left in they cluster separately, get mislabeled as "resting fibroblast," and inject an RBC program (HBB/HBA/ALAS2/GYPA/SLC4A1) into the STATE contrast. Thresholds are conservative defaults; adjust after looking at the distributions.

In [ ]:
# 5. QC  (also removes erythrocyte / RBC contamination)
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
HB_GENES = ["HBA1","HBA2","HBB","HBD","HBM","HBQ1","HBZ","HBE1"]
adata.var["hb"] = adata.var_names.str.upper().isin(HB_GENES)
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "hb"], percent_top=None, inplace=True)

sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_hb"],
             jitter=0.4, multi_panel=True, save="_qc.png", show=False)

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[adata.obs.pct_counts_mt < 15].copy()
adata = adata[adata.obs.n_genes_by_counts < 6000].copy()

# Erythrocytes are near-pure hemoglobin and otherwise gene-poor. Left in, they
# form a cluster that gets mislabeled "resting fibroblast" and pollutes the STATE
# contrast with an RBC program (HBB/HBA/ALAS2/GYPA/SLC4A1...). Drop them here.
n_before = adata.n_obs
adata = adata[adata.obs.pct_counts_hb < 20].copy()
print(f"removed {n_before - adata.n_obs} erythrocyte/RBC-contaminated cells")
print("after QC:", adata.shape)
print(adata.obs["sample"].value_counts())

## Step 4 — Normalize, cluster, embed

Log-normalize, HVGs, PCA, then neighbors/UMAP/Leiden. Because there are two donors, we optionally run Harmony on the donor label so compartments are defined by cell type rather than donor. The STATE signature later pools cells across donors, which is the point of using a state contrast on a 1v1 design.

In [ ]:
# 6. Normalize + embed
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key="sample")
# keep hemoglobin + mito out of the variable-gene set so residual ambient
# contamination does not drive PCA / clustering
adata.var.loc[adata.var["mt"] | adata.var["hb"], "highly_variable"] = False
adata_hvg = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, n_comps=30)

use_rep = "X_pca"
try:
    sc.external.pp.harmony_integrate(adata_hvg, "sample")
    use_rep = "X_pca_harmony"
    print("Harmony integration applied.")
except Exception as e:
    print("Harmony skipped (", e, ") - using raw PCA.")

sc.pp.neighbors(adata_hvg, n_neighbors=15, use_rep=use_rep)
sc.tl.leiden(adata_hvg, resolution=1.0, key_added="leiden")
sc.tl.umap(adata_hvg)

# carry embedding/labels back to the full-gene object
adata.obs["leiden"] = adata_hvg.obs["leiden"].values
adata.obsm["X_umap"] = adata_hvg.obsm["X_umap"]
sc.pl.umap(adata, color=["leiden", "sample"], save="_clusters.png", show=False)
print(adata.obs["leiden"].value_counts())

## Step 5 — Annotate compartments (marker scoring)

Score canonical marker sets per cell, then label each Leiden cluster by its dominant compartment. Never pool fibroblast / macrophage / endothelial into one signature.

In [ ]:
# 7. Compartment annotation
markers = {
    "Fibroblast":  ["COL1A1","COL1A2","COL3A1","DCN","LUM","PDGFRA","PDGFRB"],
    "Myofibroblast":["ACTA2","TAGLN","POSTN","FN1","TNC","THBS1","COMP"],
    "Macrophage":  ["CD68","LYZ","AIF1","CD163","C1QA","C1QB","SPP1","MRC1"],
    "Endothelial": ["PECAM1","VWF","CLDN5","CDH5","FLT1"],
    "Tcell":       ["PTPRC","CD3D","CD3E","IL7R"],
    "SmoothMuscle":["MYH11","MYL9","DES"],
    "Erythrocyte": ["HBB","HBA1","HBA2","ALAS2","GYPA","SLC4A1","AHSP","SLC25A37"],
}
for k, genes in markers.items():
    present = [g for g in genes if g in adata.raw.var_names]
    sc.tl.score_genes(adata, present, score_name=f"score_{k}", use_raw=True)

score_cols = [f"score_{k}" for k in markers]
cluster_scores = adata.obs.groupby("leiden")[score_cols].mean()
cluster_label = cluster_scores.idxmax(axis=1).str.replace("score_", "")
adata.obs["compartment"] = adata.obs["leiden"].map(cluster_label).astype(str)

print(cluster_scores.round(2))
print("\nCompartment assignment per cluster:")
print(cluster_label)
sc.pl.umap(adata, color=["compartment"], save="_compartment.png", show=False)
print(adata.obs["compartment"].value_counts())

## Step 6 — PRIMARY signature: within-fibroblast STATE contrast

Subset to fibroblasts (including the myofibroblast label), subcluster, score the activated/myofibroblast program, then contrast the **activated fibroblast** subclusters against the **resting fibroblast** subclusters. This pools cells across both donors, so the signature reflects fibroblast state rather than the 1v1 donor difference. The ranked gene table is the input to the LINCS L1000 reversal step.

In [ ]:
# 8. Fibroblast STATE contrast
fib = adata[adata.obs["compartment"].isin(["Fibroblast","Myofibroblast"])].copy()
print("fibroblast-compartment cells:", fib.shape[0])

# subcluster fibroblasts
fh = fib[:, fib.var_names.isin(adata_hvg.var_names)].copy()
sc.pp.scale(fh, max_value=10)
sc.tl.pca(fh, n_comps=20)
sc.pp.neighbors(fh, n_neighbors=15)
sc.tl.leiden(fh, resolution=0.6, key_added="fib_sub")
fib.obs["fib_sub"] = fh.obs["fib_sub"].values

act_program = ["ACTA2","TAGLN","POSTN","FN1","TNC","THBS1","COMP","COL1A1","COL3A1"]
act_program = [g for g in act_program if g in fib.raw.var_names]
sc.tl.score_genes(fib, act_program, score_name="activated_score", use_raw=True)

sub_scores = fib.obs.groupby("fib_sub")["activated_score"].mean().sort_values()
print("Activated score by fibroblast subcluster:\n", sub_scores.round(3))

lo = sub_scores.index[0]                 # lowest = resting
hi = sub_scores.index[-1]                # highest = activated/myofibroblast
fib.obs["fib_state"] = "intermediate"
fib.obs.loc[fib.obs["fib_sub"] == lo, "fib_state"] = "resting"
fib.obs.loc[fib.obs["fib_sub"] == hi, "fib_state"] = "activated"
print("\n", fib.obs["fib_state"].value_counts())

# sanity check: the "activated" group must be higher in myofibroblast markers,
# otherwise the split is driven by contamination, not fibroblast state
chk = [g for g in ["ACTA2","TAGLN","POSTN","FN1","CCN2","COL1A1"] if g in fib.raw.var_names]
_e = sc.get.obs_df(fib, keys=chk + ["fib_state"], use_raw=True)
_m = _e[_e["fib_state"].isin(["activated","resting"])].groupby("fib_state")[chk].mean().round(3)
print("\nMean activation-marker expression by state (activated should be higher):")
print(_m)
if {"activated","resting"} <= set(_m.index) and (_m.loc["activated"] >= _m.loc["resting"]).sum() < max(1, len(chk)//2):
    print("WARNING: 'activated' group is NOT clearly higher in myofibroblast markers -"
          " inspect clusters before trusting this signature.")

st = fib[fib.obs["fib_state"].isin(["activated","resting"])].copy()
sc.tl.rank_genes_groups(st, "fib_state", groups=["activated"], reference="resting",
                        method="wilcoxon", use_raw=True)
res = sc.get.rank_genes_groups_df(st, group="activated")
res.to_csv("out/LF_STATE_activated_vs_resting.csv", index=False)
print("\nTop UP in activated fibroblasts:")
print(res.sort_values("scores", ascending=False).head(15)[["names","logfoldchanges","pvals_adj","scores"]])
print("\nSaved out/LF_STATE_activated_vs_resting.csv")

## Step 7 — SECONDARY (fragile): HLF vs NLF within fibroblasts

Kept only as a sanity check. With one donor per group this contrast is confounded by donor identity and must **not** be used as the signature or reported as a disease effect. If the state and disease contrasts disagree, trust the state contrast.

In [ ]:
# 9. Disease contrast within fibroblasts (fragile, donor-confounded)
if fib.obs["sample"].nunique() > 1:
    sc.tl.rank_genes_groups(fib, "sample", groups=["HLF"], reference="NLF",
                            method="wilcoxon", use_raw=True)
    dres = sc.get.rank_genes_groups_df(fib, group="HLF")
    dres.to_csv("out/LF_DISEASE_HLF_vs_NLF_fibroblasts_FRAGILE.csv", index=False)
    # concordance with the STATE contrast
    m = res.merge(dres, on="names", suffixes=("_state","_disease"))
    rho = m[["scores_state","scores_disease"]].corr(method="spearman").iloc[0,1]
    print(f"Spearman rho(STATE vs DISEASE gene scores) = {rho:.3f}  (context only, n=1v1)")
    print("Saved out/LF_DISEASE_HLF_vs_NLF_fibroblasts_FRAGILE.csv")
else:
    print("Only one sample present - disease contrast skipped.")

## Step 8 — Export the reversal signature + next steps

Emit the top up/down genes (HGNC symbols) from the **STATE** contrast as the query signature for LINCS L1000 / Connectivity Map reversal. The reversal-scoring step (separate) applies the program's standard guards:

- **viability + matrix-preservation penalty** — reject compounds whose "reversal" is just cytotoxicity or transcriptional shutdown of fibroblasts / matrix-homeostasis genes;
- **local-delivery feasibility filter** — favor candidates realistic for epidural / intra-ligamentous depot delivery;
- **novelty filter** — exclude compounds already named for LF (rapamycin, rolipram, decorin, cyclopamine, N-acetylcysteine, 2-DG, MSC-exosome miRNA); flag clean, novel-for-LF hits (e.g. pirfenidone, nintedanib, metformin, senolytics);
- **replication gate** — require the same candidate/mechanism to reverse the signature in a second independent LF cohort before anything advances to a filing.

In [ ]:
# 10. Build and save the query signature
# Scrub ribosomal / translation / mito / hemoglobin / immunoglobulin / housekeeping
# genes: they are technical noise (partly a UMI-depth axis) and dilute L1000 reversal.
import json, re as _re
N = 150
HB_ = {"HBB","HBA1","HBA2","HBD","HBM","HBQ1","HBZ","HBE1"}
HOUSE_ = {"MALAT1","NEAT1","XIST","ACTB","ACTG1","TMSB4X","TMSB10","B2M","EEF1A1",
          "EEF1A2","EEF2","GAPDH","FTL","FTH1","NPM1","TPT1","NACA","MT2A","MT1X"}
def _is_noise(g):
    u = g.upper()
    if u in HB_ or u in HOUSE_: return True
    if u.startswith(("RPS","RPL","MRPS","MRPL","MT-","EIF")): return True
    if _re.match(r"^IG[HKL][VDJCG]", u): return True
    return False

res_clean = res[~res["names"].map(_is_noise)].copy()
n_dropped = len(res) - len(res_clean)
up = (res_clean[(res_clean["logfoldchanges"] > 0.5) & (res_clean["pvals_adj"] < 0.05)]
      .sort_values("scores", ascending=False)["names"].head(N).tolist())
dn = (res_clean[(res_clean["logfoldchanges"] < -0.5) & (res_clean["pvals_adj"] < 0.05)]
      .sort_values("scores", ascending=True)["names"].head(N).tolist())

sig = {"signature": "LF_activated_vs_resting_fibroblast",
       "source": "GSE294458 (pilot, 1 HLF vs 1 NLF)",
       "scrubbed_noise_genes": int(n_dropped),
       "n_up": len(up), "n_down": len(dn), "up": up, "down": dn}
with open("out/LF_reversal_signature.json", "w") as fh:
    json.dump(sig, fh, indent=2)

print(f"scrubbed {n_dropped} ribosomal/housekeeping/mito/Ig genes from the DE table")
print(f"UP genes ({len(up)}):", ", ".join(up[:20]), "...")
print(f"\nDOWN genes ({len(dn)}):", ", ".join(dn[:20]), "...")
print("\nSaved out/LF_reversal_signature.json")
print("\nPILOT COMPLETE - do not advance to filing until reproduced in a 2nd LF cohort.")